In [444]:
import pandas as pd
from PIL import Image
from IPython.display import display, clear_output
from IPython.display import Image as DisplayImage
import io
import requests
from IPython.display import display, clear_output
import ipywidgets as widgets
import json
#audio imports 
import requests
import json
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from IPython.display import Audio
import base64


In [445]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* DAWN - soft peach */
.widget-toggle-buttons .widget-toggle-button:nth-child(1) {
    background-color: #E8BCA8 !important;
    color: #493A35 !important;
}

/* DAY - sandy yellow */
.widget-toggle-buttons .widget-toggle-button:nth-child(2) {
    background-color: #E8D39D !important;
    color: #493A35 !important;
}

/* DUSK - dusty rose */
.widget-toggle-buttons .widget-toggle-button:nth-child(3) {
    background-color: #B87985 !important;
    color: white !important;
}

/* NIGHT - muted lavender */
.widget-toggle-buttons .widget-toggle-button:nth-child(4) {
    background-color: #9990B4 !important;
    color: white !important;
}

/* Highlight the selected time */
.widget-toggle-buttons .widget-toggle-button.mod-active {
    border: 3px solid #51443F !important;
    box-shadow: inset 0 0 0 1px white !important;
    font-weight: bold !important;
}

/* EXPLORE / CONTINUE EXPLORING - sage green */
.widget-button.custom-desert-button {
    background-color: #A3B18A !important;
    color: white !important;
    border: none !important;
}
</style>
"""))

In [446]:
# Load credentials from your service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

In [447]:
# Fetch an access token (this is what gcloud auth print-access-token returns)
creds.refresh(Request())
token = creds.token

In [448]:
with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

In [449]:
def get_audio(prompt):

    creds.refresh(Request())
    token = creds.token

    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
        "model": "lyria-3-clip-preview",
        "input": [{"type": "text", "text": prompt}]
    }

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, json=payload, timeout=120)
    response.raise_for_status()

    return response.json()["outputs"][2]["data"]

In [450]:
# x = get_audio(
#     "Mysterious ambient instrumental music for a moonlit desert, "
#     "soft atmospheric textures, gentle percussion, no vocals"
# )

# mp3_bytes = base64.b64decode(x)

# display(Audio(data=mp3_bytes, autoplay=False))

In [451]:
music_cache = {}

def load_chapter_music():

    # choose music based on the current location
    if current_location == "Moon Hollow":
        prompt = (
            "Mysterious ambient instrumental music for a moonlit desert, "
            "soft atmospheric textures, gentle percussion, no vocals"
        )

    elif current_location == "Glass Dunes":
        prompt = (
            "Dreamlike ambient instrumental music for golden desert dunes, "
            "shimmering textures, warm atmosphere, no vocals"
        )

    else:
        return

    with audio_output:
        clear_output(wait=True)

        print("The desert is waking up...")
        print("Generating your soundtrack. Please wait until we're ready.")

        try:
            # generate music if we haven't already
            if current_location not in music_cache:
                x = get_audio(prompt)
                music_cache[current_location] = base64.b64decode(x)

            # clear the loading message once music is ready
            audio_output.clear_output(wait=True)

            with audio_output:
                print("Your soundtrack is ready!")
                display(Audio(
                    data=music_cache[current_location],
                    autoplay=True
                ))

        except Exception as error:
            audio_output.clear_output(wait=True)
            with audio_output:
                print("Audio generation failed:", error)

In [452]:
gals_df = pd.read_csv('data/desert.csv')

In [453]:
print(gals_df.iloc[0]["location"])
#it works!

Glass Dunes


In [454]:
display(gals_df.head())

,location,discovery,category,time_of_day,description,story,asset
0,Glass Dunes,Desert glass,Geology,Day,A smooth fragment of glass lies among the sand.,Sand can change form under extraordinary heat....,desert_glass.png
1,Glass Dunes,Ripple marks,Geology,Dawn,Parallel ridges stretch across the dune surface.,The wind has been writing here all night. By m...,ripple_marks.png
2,Glass Dunes,Beetle tracks,Wildlife,Dawn,Tiny paired tracks cross an otherwise untouche...,Something small has crossed this enormous plac...,beetle_tracks.png
3,Glass Dunes,Buried seed,Botany,Dusk,A seed rests just beneath the dry surface.,It has not disappeared. It is waiting for the ...,buried_seed.png
4,Glass Dunes,Wind-polished stone,Geology,Day,One side of a small stone is worn smooth by bl...,Change can happen so slowly that no one notice...,polished_stone.png


In [455]:
#i need to filter and make a dataset because 2 things are super important in my game setup, location and time of day and then their relation to each other 
# the game is 2 chapters so far so I'm goign to be using that to guide the filtering

moon_hollow_df = gals_df[gals_df["location"] == "Moon Hollow"]
glass_dunes_df = gals_df[gals_df["location"] == "Glass Dunes"]

# start the game in chapter 1
current_chapter = 1
current_location = "Moon Hollow"
current_location_df = moon_hollow_df

In [456]:
# i want to make a list for everything collected but i dont want to allow duplicates so im going to use a set, to start it's going to be empty
discovered = set()
# when the game first starts, the player hasn't encountered anything yet, so im setting the current discovery to none
current_discovery = None


In [457]:
#player chooses the time then starts the game 
time_selector = widgets.ToggleButtons(
    options=["Dawn", "Day", "Dusk", "Night"],
    value="Day",
    description="Time:",
    button_style=""
)

In [458]:
# this function shows the Moon Hollow background
# based on the time selected by the player

def show_background():
    

    selected_time = time_selector.value.lower()

#make sure we format this correctly
    location_name = current_location.lower().replace(" ", "_")

    # create the path to the background
    image_path = f"assets/{location_name}_{selected_time}.png"

    display(DisplayImage(filename=image_path, width=700))

In [459]:
# 3 buttons - to start the game, to explore the desert and to investigate an object --> invesigate an object is conditional though
start_button = widgets.Button(description="Start Game")
explore_button = widgets.Button(description="Explore")
investigate_button = widgets.Button(description="Investigate")
#new addition for chapter control
next_chapter_button = widgets.Button(
    description="Enter Chapter 2: Glass Dunes",
    button_style="success"
)

next_chapter_button.layout.display = "none"

audio_button = widgets.Button(
    description="Generate Music"
)

audio_output = widgets.Output()


In [460]:
# making investigate button conditional (is there a better way to do this? like a state tracker) 
investigate_button.disabled = True


In [461]:
#game text area
output = widgets.Output()

# tracking the number of discoveries
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(moon_hollow_df),
    description="Found:"
)
progress.style.bar_color = "#A3B18A"
# group all the game controls together
game_buttons = widgets.VBox([
    time_selector,
    explore_button,
    investigate_button,
    progress, 
    next_chapter_button
])

# hide the controls until the player starts
game_buttons.layout.display = "none"

# hide investigate until the player finds something
investigate_button.layout.display = "none"

In [462]:
#i decided to break 
def gameControls(button): 
    print("DEBUG: Start button was clicked")
    # hide the start button
    start_button.layout.display = "none"

    # show the game controls
    game_buttons.layout.display = "flex"
    with output:
            clear_output()

            show_background()
            print("Welcome to BUFFALO GALS DESERT")
            print()
            print("Welcome to Moon Hollow.")
            print()
            print("The desert appears empty, but not")
            print("everything reveals itself in daylight.")
            print()
            print("Change the time of day and explore.")
            print("Discover the things hidden in the desert.")
    load_chapter_music()

            # display(time_selector)
            # display(explore_button)
            # display(investigate_button)
            # display(progress)
        
    

In [463]:
#function for everytime the player wants to explore 
def explore(button):
    print("DEBUG: Explore function is running")

    global current_discovery #this is telling this function NOT to create a local vairiable with the same name as the global one we defined but to use the global one since it's also our tracker 

    # Get the value for the selected time
    selected_time = time_selector.value

    # get discoveries available at this time depending on location (ch1 or ch2) 
    available = current_location_df[
        current_location_df["time_of_day"] == selected_time
            ]

    # filter out the discoveries already found
    remaining = []
    
    for index, row in available.iterrows():
        if row["discovery"] not in discovered:
            remaining.append(index)
    
    available = available.loc[remaining]

    with output:
        clear_output()
        show_background()

        print(current_location.upper())
        print("Time:", selected_time)
        print() #newline --> is there an easier way to newline 

        
        #based on the dataset, check the discoveries, if none are available .empty will return ture 
        if available.empty:
            explore_button.layout.display = "block"
            investigate_button.layout.display = "none"
            current_discovery = None
            investigate_button.disabled = True

            print("Nothing new reveals itself here.")
            print("Try another time of day.")

        else: #if there are discoveries at that time 
            # Choose a random discovery
            current_discovery = available.sample().iloc[0]
            
            # show investigate instead of explore
            explore_button.layout.display = "none"
            investigate_button.layout.display = "block"
            investigate_button.disabled = False

            print("You notice something unusual...")
            print()
            print("Clue:", current_discovery["category"])
            print()
            print("Click Investigate to learn more.")
            

In [464]:
#when user click on an object and needs to investigate 
def investigate(button):

    global current_discovery #again, using and manipulating the same variable prev defined 
    
    #if they're not currently exploring anything just leave the function  
    if current_discovery is None:
        return

    # otherwise, get the name of the thing being discovered 
    discovery_name = current_discovery["discovery"]

    # count how many discoveries the player has collected
    # and update the progress bar with that number
    discovered.add(discovery_name)
    progress.value = len(discovered)

    with output:
        clear_output()
        show_background()
        #get relevent info about the object from the CSV such as description and story 
        print("NEW DISCOVERY!")
        print()
        # display the discovery's name
        print(discovery_name.upper())
        print()
        
        # get the image filename from our CSV
        asset_name = current_discovery["asset"]
        
        # find that image in our assets folder
        asset_path = f"assets/{asset_name}"
        
        # display the object the player discovered
        display(DisplayImage(filename=asset_path, width=250))
        
        # display the description from our CSV
        print(current_discovery["description"])
        print()
        print("--- THE STORY ---")
        print(current_discovery["story"])
        print()
        
       # i also want to show how many discoveries the user has collected and so len(discovered) counts the collected items and len(moon_hollow) counts the total discoveries available at this level
        print("Discovered:", len(discovered), "/", len(current_location_df))

        if len(discovered) == len(current_location_df): # check if the player has collected every discovery in moon hollow

            print()
            print("CHAPTER COMPLETE!")
            print()

            if current_chapter==1:
                print ("You've discovered everything in Moon hollow!!")
                print ("Beyond the hollow, the Glass Dunes await")

                next_chapter_button.layout.display="block"

            elif current_chapter==2: 
                print ("You've discovered everything in Glass Dunes!")
                print ("The desert still has more stories to tell")

      # reset the current discoverysh
        current_discovery = None
        investigate_button.disabled = True
        investigate_button.layout.display = "none"

        if len(discovered) == len(current_location_df):
    
            # stop exploring once the chapter is complete
            explore_button.layout.display = "none"
            time_selector.layout.display = "none"
    
        else:
            # otherwise, continue exploring
            explore_button.description = "Continue Exploring"
            explore_button.layout.display = "block"
        

In [465]:
# this function runs whenever the player changes the time of day

def change_time(change):

    global current_discovery

    # reset the current discovery because the player has changed the time
    current_discovery = None
    investigate_button.disabled = True

    with output:
        clear_output()
        show_background()
        print("The desert shifts into", change["new"].lower() + ".")
        print()
        print("Click Explore to see what's here.")
        # reset buttons when the time changes
        investigate_button.layout.display = "none"
        explore_button.layout.display = "block"
        explore_button.description = "Explore"

In [466]:
def start_chapter_two(button):

    global current_chapter, current_location
    global current_location_df, current_discovery, discovered

    # move to chapter 2
    current_chapter = 2
    current_location = "Glass Dunes"
    current_location_df = glass_dunes_df

    # reset discoveries for the new chapter
    discovered = set()
    current_discovery = None

    # reset the progress bar for the 6 new discoveries
    progress.value = 0
    progress.max = len(current_location_df)

    # hide the chapter transition button
    next_chapter_button.layout.display = "none"

    # show the exploration controls again
    time_selector.layout.display = "block"
    explore_button.layout.display = "block"
    explore_button.description = "Explore"

    investigate_button.layout.display = "none"
    investigate_button.disabled = True

    # show the new chapter introduction
    with output:
        clear_output()
        show_background()

        print("CHAPTER 2: GLASS DUNES")
        print()
        print("You leave Moon Hollow behind.")
        print()
        print("Ahead, the dunes stretch toward the horizon.")
        print("The sand shimmers, and nothing is quite")
        print("what it first appears to be.")
        print()
        print("Choose a time of day and begin exploring.")
    load_chapter_music()


In [467]:
# connect the start button to the gameControls function
start_button.on_click(gameControls)
explore_button.on_click(explore)

investigate_button.on_click(investigate)

time_selector.observe(change_time, names="value")
next_chapter_button.on_click(start_chapter_two)


In [468]:
print("A DESERT IN TWO WORLDS")
print("Chapter 1: Moon Hollow")
print()

# title screen
display(start_button)

# controls appear after Start Game
display(game_buttons)

# background and narrative
display(output)
display(audio_output)


A DESERT IN TWO WORLDS
Chapter 1: Moon Hollow



Button(description='Start Game', style=ButtonStyle())

Output()

Output()

In [388]:
# debugging code because nothing was printing 
#time_selector.value = "Night"

# explore(None)

# print("Current discovery:", current_discovery)

In [389]:
# debugging code to remove the old connection and connect the explore button to the function
#explore_button.on_click(explore, remove=True)
#explore_button.on_click(explore)

In [390]:
#debugging to see if jupyter is communicating with my python kernels, it is not :( 
test_slider = widgets.IntSlider(
    value=5,
    min=0,
    max=10,
    description="Test:"
)

display(test_slider)

IntSlider(value=5, description='Test:', max=10)

In [391]:
print(test_slider.value)

5


In [392]:
import os

print(os.getcwd())
print(os.listdir())

/Users/wpp74634/Desktop/data-assignment-github
['.DS_Store', 'requirements.txt', 'Untitled.ipynb', 'README.md', '.gitignore', 'keys', '.ipynb_checkpoints', 'assignment5.ipynb', 'venv', '.git', 'data', 'assets', 'Game.ipynb', '01_Pokemon.ipynb']


In [393]:
print(os.listdir("assets"))

['.DS_Store', 'cooled_rock.png', 'night_flower.png', 'polished_stone.png', 'buried_seed.png', 'false_horizon.png', 'burrow.png', 'moth.png', 'moon_hollow_night.png', 'beetle_tracks.png', 'ripple_marks.png', 'moon_hollow_dawn.png', 'desert_glass.png', 'moon_hollow_dusk.png', 'glass_dunes_night.png', 'moon_hollow_day.png', 'glass_dunes_day.png', 'glass_dunes_dusk.png', 'glass_dunes_dawn.png']
